In [1]:
import os
import random
import string
import hail as hl
from ukb_utils import hail_init
import numpy as np
from ko_utils import ko
from ko_utils import samples
from ko_utils.ko import PLOF_CSQS, MISSENSE_CSQS, SYNONYMOUS_CSQS, OTHER_CSQS

In [2]:
2

2

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa006.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


# Simulate new model with 3 random moments

In [6]:
prune_hom_alt = 0.99
seed = 42
h2_nc = 0.05
h2_co = 0.15
h2_ko = 0.30
pi_nc = None
pi_co = None
pi_ko = None
K = 0.1

# Simulate absence of compound het effect
How can we simulate the absence of CH effects..
1. Randomly assign phenotypes?
2. Spike and slab model
3. Polygenic background
4. Combine (2) and (3)

In [8]:
def _get_tid(length=5):
    r"""Internal method for getting random ID string to append to field names"""
    return ''.join(random.choices(string.ascii_uppercase + string.ascii_lowercase, 
                                  k=length))

def annotate_all(mt, row_exprs={}, col_exprs={}, entry_exprs={}, global_exprs={}):
    r"""Equivalent of _annotate_all, but checks source MatrixTable of exprs"""
    exprs = {**row_exprs, **col_exprs, **entry_exprs, **global_exprs}
    for key, value in exprs.items():
        if type(value) == hl.expr_float64 or type(value) == hl.expr_int32:
            assert value._indices.source == mt, 'Cannot combine expressions from different source objects.'
    return mt._annotate_all(row_exprs, col_exprs, entry_exprs, global_exprs)

In [9]:
beta = mt.beta
popstrat=None
popstrat_var=None
genotype = mt.GT
h2 = 0.9

In [10]:
tid = _get_tid()
mt = annotate_all(mt=mt,
                 row_exprs={'beta_'+tid: beta},
                 col_exprs={} if popstrat is None else {'popstrat_'+tid: popstrat},
                 entry_exprs={'gt_'+tid: genotype.n_alt_alleles() if genotype.dtype is hl.dtype('call') else genotype})
mt = mt.filter_rows(hl.agg.stats(mt['gt_'+tid]).stdev>0)
mt = hl.experimental.ldscsim.normalize_genotypes(mt['gt_'+tid])


In [11]:
mt = mt.annotate_cols(y_no_noise=hl.agg.sum(mt['beta_'+tid] * mt['norm_gt']))
#mt.aggregate_cols(hl.agg.mean(mt.y_no_noise))
mt = mt.annotate_cols(y=mt.y_no_noise + hl.rand_norm(0, hl.sqrt(1-h2)))

In [13]:
path = 'data/permute/counts/ukb_eur_wes_200k_pLoF_damaging_missense_counts_chr21.vcf.gz'


<Float64Expression of type float64>

In [12]:
#result_ht = hl.linear_regression_rows(
 #    y=mt.y,
 #    x=mt.GT.n_alt_alleles(),
 #    covariates=[1])

In [3]:
def agg_count_calls(mt: hl.MatrixTable, phased: bool = True):
    """ Count number of phased/unphased hetz and what haplotype they reside on

    :param mt: MatrixTable with GT field
    :param mt: test for phased genotypes only
    """
    aggr = mt.aggregate_entries(hl.agg.counter(mt.GT))
    gt10 = aggr[hl.Call(alleles=[1, 0], phased = phased)]
    gt01 = aggr[hl.Call(alleles=[0, 1], phased = phased)]
    return((gt10,gt01))
    

# manually simulated with Hail

In [4]:
n = 10
mt = hl.balding_nichols_model(10, 100, 1000, reference_genome='GRCh38')
mt = mt.transmute_entries(GT = ko.set_to_phased_call(mt.GT))
mt = mt.transmute_entries(GT = ko.rand_flip_call(mt.GT))
#ko.agg_count_calls(mt)

2022-04-11 09:37:16 Hail: INFO: balding_nichols_model: generating genotypes for 10 populations, 100 samples, and 1000 variants...


In [5]:
n = 10
mt = hl.balding_nichols_model(1, 10, n, reference_genome='GRCh38')
mt = mt.transmute_entries(GT = ko.set_to_phased_call(mt.GT))
mt = hl.experimental.ldscsim.simulate_phenotypes(mt, mt.GT, h2 = 0)
#mt = mt.annotate_rows(maf = hl.min(hl.agg.call_stats(mt.GT, mt.alleles).AF))
#mt = mt.annotate_rows(ES = 2 * (mt.beta**2)*mt.maf*(1-mt.maf))
#mt = mt.annotate_rows(h2i = mt.ES * n)

2022-04-11 09:37:17 Hail: INFO: balding_nichols_model: generating genotypes for 1 populations, 10 samples, and 10 variants...


calculating phenotype


In [513]:
annotation = ["pLoF","LC","damaging_missense","other_missense","synonymous"]
weights = [0.02, 0.03, 0.05, 0.1, 0.8]
def random_gene_annotation(k, annotation = ['a','b'], weights = [0.4, 0.6]):
    return(random.choices(annotation, weights = weights, k = k))

In [514]:
df = mt.rows().drop('bn').to_pandas()
df[['csqs']] = random_annotation(n, annotation, weights)
df[['gene_id']] = 'CACNA1C'

2022-02-23 18:15:21 Hail: INFO: Coerced sorted dataset


In [515]:

ht = hl.Table.from_pandas(df)
ht = ht.annotate(
    locus = hl.parse_variant(
        hl.delimit([ht['locus.contig'],hl.str(ht['locus.position']),ht.alleles[0],ht.alleles[1]],':')
    )
)
ht = ht.key_by(ht.locus)
mt = mt.annotate_rows(
    gene_id = ht[mt.row_key].gene_id,
    csqs = ht[mt.row_key].csqs

)
mt = mt.filter_rows(hl.literal(set(['pLoF','LC','damaging_missense'])).contains(mt.csqs))
mt.count()

2022-02-23 18:15:23 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:23 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:23 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:24 Hail: INFO: Ordering unsorted dataset with network shuffle


(108, 10)

In [516]:
mt_gene = (mt.group_rows_by(mt.gene_id)
            .aggregate(gts=hl.agg.collect(mt.GT)))

In [517]:
mt_gene = ko.sum_gts_entries(mt_gene)
expr_pko = ko.calc_prob_ko(mt_gene.hom_alt, mt_gene.phased, mt_gene.unphased)
expr_ko = ko.annotate_knockout(mt_gene.hom_alt, expr_pko)
mt_gene = mt_gene.annotate_entries(
            pKO = expr_pko,
            knockout = expr_ko)
mt_gene.knockout.show()

2022-02-23 18:15:31 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:31 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:32 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:33 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-02-23 18:15:33 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:33 Hail: INFO: Coerced sorted dataset
2022-02-23 18:15:34 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-02-23 18:15:36 Hail: INFO: Coerced sorted dataset


,,,,,
,0,1,2,3,4
gene_id,knockout,knockout,knockout,knockout,knockout
str,str,str,str,str,str
"""CACNA1C""","""Homozygote""","""Homozygote""","""Homozygote""","""Homozygote""","""Homozygote"""


(182, 500)

In [273]:
mt_gene = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id)
            .aggregate(gts=hl.agg.collect(mt.GT)))

In [275]:
mt_gene = ko.sum_gts_entries(mt_gene)
expr_pko = ko.calc_prob_ko(mt_gene.hom_alt, mt_gene.phased, mt_gene.unphased)
expr_ko = ko.annotate_knockout(mt_gene.hom_alt, expr_pko)
mt_gene = mt_gene.annotate_entries(
            pKO = expr_pko,
            knockout = expr_ko)

In [508]:
mt_gene.pKO.show()

2022-02-23 18:10:40 Hail: INFO: Coerced sorted dataset
2022-02-23 18:10:40 Hail: INFO: Coerced sorted dataset
2022-02-23 18:10:41 Hail: INFO: Coerced sorted dataset
2022-02-23 18:10:41 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-02-23 18:10:42 Hail: INFO: Coerced sorted dataset


,,,,,
,0,1,2,3,4
gene_id,pKO,pKO,pKO,pKO,pKO
str,float64,float64,float64,float64,float64
"""CACNA1C""",1.00e+00,1.00e+00,1.00e+00,1.00e+00,1.00e+00
